# 🚀 Huấn Luyện NLLB Chuyên Biệt Trung - Việt (LoRA v2 - Khắc Phục Ngữ Pháp & Tên Riêng)

### 📌 Cơ Chế Huấn Luyện Mới Nối Tiếp Từ Bản 8.000 Steps:
- **Base Model**: Sử dụng trực tiếp mô hình bạn đã merge ở Step 8000 (`/content/drive/MyDrive/nllb_zh_vi_final_merged`). Nền tảng 8.000 bước được giữ nguyên 100%.
- **Kích hoạt FFN (`fc1`, `fc2`)**: Mở khóa 2 lớp Feed-Forward Networks để chứa tri thức ngữ pháp tiếng Việt và ánh xạ tên riêng phương Tây/Nhật/Hàn.
- **Tăng Rank (r=32, alpha=64)**: Tăng gấp đôi dung lượng bộ nhớ biểu diễn từ vựng.
- **Beam Search Decoding**: Đánh giá văn phong với `num_beams=4`, loại bỏ triệt để hiện tượng lặp từ và câu cụt của Greedy Search.

---
### 🟩 CELL 1: Kết nối Google Drive & Kiểm tra GPU

In [ ]:
# CELL 1: Mount Google Drive & Kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

---
### 🟩 CELL 2: Cài đặt Môi Trường (Khắc phục lỗi xung đột torchao)

In [ ]:
# CELL 2: Gỡ bỏ torchao 0.10.0 cũ và cài đặt thư viện tương thích
!pip uninstall -y torchao
!pip install -q -U "transformers>=4.40.0" "datasets>=2.19.0" "peft>=0.11.0" accelerate sentencepiece sacrebleu

---
### 🟩 CELL 3: Chuẩn Bị Dữ Liệu & Khởi Tạo Script Huấn Luyện Mới

In [ ]:
# CELL 3: Đồng bộ Dataset & Tải Script Train mới nhất
import os

# Kiểm tra dataset
if not os.path.exists('/content/dataset_full_zh_vi'):
    BUNDLE_PATH = '/content/drive/MyDrive/nllb_zh_vi_bundle.zip'
    if os.path.exists(BUNDLE_PATH):
        print('Đang giải nén bộ dataset từ Google Drive...')
        !unzip -q -o '/content/drive/MyDrive/nllb_zh_vi_bundle.zip' -d /content/
        print('✅ Hoàn tất giải nén!')
    else:
        print('⚠️ Không tìm thấy nllb_zh_vi_bundle.zip trên Google Drive! Kiểm tra lại đường dẫn.')
else:
    print('✅ Thư mục dataset_full_zh_vi đã sẵn sàng!')

---
### 🟩 CELL 4: Khởi Tạo Script Huấn Luyện `train_colab_zh_vi.py` (Tự động ghi vào Colab)

In [ ]:
# CELL 4: Ghi trực tiếp file train_colab_zh_vi.py nâng cấp lên Colab (Không cần upload thủ công)
code = '''"""
Modular & Fault-Tolerant LoRA Fine-Tuning Script for NLLB-Pruned Chinese-Vietnamese (zh_vi).
Optimized for Google Colab (Free T4 GPU / 16GB VRAM) and spotty networks.

Key Features:
- Chunked training (e.g. 500 - 1,000 steps per stage) with explicit max_steps.
- Auto-sync checkpoints to Google Drive (/content/drive/MyDrive/nllb_zh_vi_checkpoints).
- Auto-resume from the latest checkpoint on Google Drive upon restart or disconnection.
- 100% compatible with Transformers v4.40 - v5.x (dynamic argument inspection).
- In-flight benchmark generation on Golden Test sets at the end of each stage.
"""

import argparse
import inspect
import json
import os
import shutil
import sys
import time
import torch

# ---------------------------------------------------------------------------
# Hotfix: Neutralize Colab's pre-installed torchao 0.10.0 bug in PEFT
# In PEFT, is_torchao_available() incorrectly raises ImportError instead of
# returning False when torchao < 0.16.0 is installed in Python environment.
# ---------------------------------------------------------------------------
import sys
sys.modules["torchao"] = None

try:
    import peft.import_utils
    peft.import_utils.is_torchao_available = lambda: False
except Exception:
    pass
try:
    import peft.tuners.lora.torchao as _torchao_dispatch
    _torchao_dispatch.is_torchao_available = lambda: False
except Exception:
    pass

from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")


def sync_to_drive(local_dir: str, drive_dir: str):
    if not os.path.exists(drive_dir):
        os.makedirs(drive_dir, exist_ok=True)
    print(f"\\n[Drive Sync] Syncing checkpoints from {local_dir} -> {drive_dir}...")
    for item in os.listdir(local_dir):
        s_path = os.path.join(local_dir, item)
        d_path = os.path.join(drive_dir, item)
        if os.path.isdir(s_path):
            if os.path.exists(d_path):
                shutil.rmtree(d_path, ignore_errors=True)
            shutil.copytree(s_path, d_path)
        else:
            shutil.copy2(s_path, d_path)
    print("[Drive Sync] Sync completed successfully!\\n")


def find_latest_checkpoint(checkpoint_dir: str) -> str | None:
    if not os.path.exists(checkpoint_dir):
        return None
    ckpts = [
        os.path.join(checkpoint_dir, d)
        for d in os.listdir(checkpoint_dir)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(checkpoint_dir, d))
    ]
    if not ckpts:
        return None
    # Sort by step number
    ckpts.sort(key=lambda x: int(x.split("-")[-1]))
    return ckpts[-1]


def evaluate_sample_sentences(model, tokenizer, test_sentences, src_lang="zho_Hans", tgt_lang="vie_Latn"):
    tokenizer.src_lang = src_lang
    tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    print("\\n" + "=" * 80)
    print("IN-FLIGHT BENCHMARK EVALUATION (Beam Search num_beams=4)")
    print("=" * 80)
    model.eval()
    for zh, desc in test_sentences:
        inp = tokenizer(zh, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inp,
                forced_bos_token_id=tgt_id,
                max_new_tokens=64,
                num_beams=4,
                no_repeat_ngram_size=3,
                repetition_penalty=1.15,
            )
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"[{desc}]")
        print(f"  ZH  : {zh}")
        print(f"  PRED: {pred}")
        print("-" * 80)


def main():
    parser = argparse.ArgumentParser(description="Fault-tolerant LoRA training for NLLB zh_vi.")
    parser.add_argument("--model-path", type=str, default="/content/nllb_pruned_zh_vi")
    parser.add_argument("--train-file", type=str, default="/content/dataset_full_zh_vi/train.jsonl")
    parser.add_argument("--eval-file", type=str, default="/content/dataset_full_zh_vi/eval.jsonl")
    parser.add_argument("--output-dir", type=str, default="/content/nllb_zh_vi_checkpoints")
    parser.add_argument("--drive-dir", type=str, default="/content/drive/MyDrive/nllb_zh_vi_checkpoints")
    parser.add_argument("--max-steps", type=int, default=500, help="Steps for this training chunk")
    parser.add_argument("--batch-size", type=int, default=16, help="Per device batch size")
    parser.add_argument("--accum-steps", type=int, default=2, help="Gradient accumulation steps")
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--save-steps", type=int, default=250)
    parser.add_argument("--lora-r", type=int, default=32)
    parser.add_argument("--lora-alpha", type=int, default=64)
    parser.add_argument("--target-modules", nargs="+", default=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"])
    parser.add_argument("--max-length", type=int, default=256)
    args = parser.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Training Device: {device.upper()}")

    # 1. Check if Google Drive has prior checkpoints to resume
    resume_checkpoint = None
    if os.path.exists(args.drive_dir):
        drive_latest = find_latest_checkpoint(args.drive_dir)
        if drive_latest:
            print(f"Found existing checkpoint on Drive: {drive_latest}")
            os.makedirs(args.output_dir, exist_ok=True)
            local_target = os.path.join(args.output_dir, os.path.basename(drive_latest))
            if not os.path.exists(local_target):
                print(f"Restoring checkpoint from Drive to local disk...")
                shutil.copytree(drive_latest, local_target)
            resume_checkpoint = local_target
            print(f"Resume target ready: {resume_checkpoint}")

    if not resume_checkpoint:
        resume_checkpoint = find_latest_checkpoint(args.output_dir)

    # 2. Fast Local Cache for Model Path (if loaded from Google Drive)
    model_path = args.model_path
    if model_path.startswith("/content/drive/") and os.path.exists(model_path):
        local_base = os.path.join("/content", os.path.basename(model_path.rstrip("/")))
        if not os.path.exists(local_base):
            print(f"Copying base model from Drive to local SSD for fast training: {local_base}...")
            shutil.copytree(model_path, local_base)
        model_path = local_base

    # 3. Load Tokenizer & Model
    print(f"Loading tokenizer from {model_path}...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    print(f"Tokenizer vocab size: {len(tokenizer):,}")

    print(f"Loading base pruned model in FP16 from {model_path}...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if device == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
    )

    if device == "cuda":
        base_model = base_model.to("cuda")

    # 4. Configure LoRA
    if resume_checkpoint and os.path.exists(os.path.join(resume_checkpoint, "adapter_config.json")):
        print(f"Loading LoRA adapter weights from checkpoint: {resume_checkpoint}...")
        model = PeftModel.from_pretrained(base_model, resume_checkpoint, is_trainable=True)
    else:
        print(f"Configuring new LoRA adapter (r={args.lora_r}, alpha={args.lora_alpha}, modules={args.target_modules})...")
        lora_config = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            inference_mode=False,
            r=args.lora_r,
            lora_alpha=args.lora_alpha,
            lora_dropout=0.05,
            target_modules=args.target_modules,
            bias="none",
        )
        model = get_peft_model(base_model, lora_config)

    model.print_trainable_parameters()

    # 4. Load & Preprocess Dataset
    print(f"Loading dataset JSONL from {args.train_file}...")
    dataset = load_dataset(
        "json",
        data_files={"train": args.train_file, "validation": args.eval_file},
    )
    print(f"Train samples: {len(dataset['train']):,}, Eval samples: {len(dataset['validation']):,}")

    max_len = args.max_length

    def preprocess_batch(batch):
        inp_ids, lbl_ids = [], []
        tokenizer.src_lang = "zho_Hans"
        tokenizer.tgt_lang = "vie_Latn"
        for src, tgt in zip(batch["source"], batch["target"]):
            inp = tokenizer(src, max_length=max_len, truncation=True)
            out = tokenizer(text_target=tgt, max_length=max_len, truncation=True)
            inp_ids.append(inp["input_ids"])
            lbl_ids.append(out["input_ids"])
        return {"input_ids": inp_ids, "labels": lbl_ids}

    print("Tokenizing train and eval datasets...")
    tokenized_dataset = dataset.map(
        preprocess_batch,
        batched=True,
        batch_size=2000,
        remove_columns=dataset["train"].column_names,
        desc="Tokenizing",
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        pad_to_multiple_of=8 if device == "cuda" else None,
    )

    # 5. Build Training Arguments with Dynamic Compatibility
    current_step = 0
    if resume_checkpoint:
        try:
            current_step = int(os.path.basename(resume_checkpoint).split("-")[-1])
        except Exception:
            current_step = 0

    target_max_steps = args.max_steps
    if target_max_steps <= current_step:
        target_max_steps = current_step + args.max_steps
        print("\\n" + "=" * 80)
        print(f"[Smart Step Calculation] Checkpoint hiện tại: {current_step} steps.")
        print(f"[Smart Step Calculation] Tự động cộng dồn: Huấn luyện thêm +{args.max_steps} steps.")
        print(f"[Smart Step Calculation] Tổng mục tiêu mới: {target_max_steps} steps.")
        print("=" * 80 + "\\n")
    else:
        print("\\n" + "=" * 80)
        print(f"[Smart Step Calculation] Checkpoint hiện tại: {current_step} steps.")
        print(f"[Smart Step Calculation] Mục tiêu tổng thể: {target_max_steps} steps (sẽ chạy thêm +{target_max_steps - current_step} steps).")
        print("=" * 80 + "\\n")

    sig_args = inspect.signature(Seq2SeqTrainingArguments.__init__)
    args_kwargs = {
        "output_dir": args.output_dir,
        "per_device_train_batch_size": args.batch_size,
        "per_device_eval_batch_size": args.batch_size,
        "gradient_accumulation_steps": args.accum_steps,
        "learning_rate": args.lr,
        "fp16": (device == "cuda"),
        "logging_steps": 50,
        "save_steps": args.save_steps,
        "save_total_limit": 2,
        "max_steps": target_max_steps,
        "report_to": "none",
    }
    if "predict_with_generate" in sig_args.parameters:
        args_kwargs["predict_with_generate"] = False
    if "warmup_steps" in sig_args.parameters:
        args_kwargs["warmup_steps"] = 100

    training_args = Seq2SeqTrainingArguments(**args_kwargs)

    # 6. Build Trainer with Dynamic Compatibility
    trainer_kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": tokenized_dataset["train"],
        "eval_dataset": tokenized_dataset["validation"],
        "data_collator": data_collator,
    }
    sig_trainer = inspect.signature(Seq2SeqTrainer.__init__)
    if "processing_class" in sig_trainer.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Seq2SeqTrainer(**trainer_kwargs)

    # 7. Start Training
    print(f"\\nStarting training chunk (Target: {target_max_steps} total steps)...")
    if resume_checkpoint:
        print(f"Resuming training from checkpoint: {resume_checkpoint}")
        trainer.train(resume_from_checkpoint=resume_checkpoint)
    else:
        trainer.train()

    # 8. Save Final Model & Adapters
    print(f"\\nSaving adapter weights to {args.output_dir}...")
    model.save_pretrained(args.output_dir)
    tokenizer.save_pretrained(args.output_dir)

    # 9. Sync to Google Drive
    if os.path.exists("/content/drive/MyDrive"):
        sync_to_drive(args.output_dir, args.drive_dir)

    # 10. Run In-Flight Benchmark Test
    test_samples = [
        ("白小纯深吸一口气，开始运转长生功。", "Xianxia Cultivation"),
        ("克莱恩·莫雷蒂站在贝克兰德的雨夜街头，轻轻按住头顶的黑色礼帽。", "Lord of the Mysteries (Western Name)"),
        ("【系统提示】：玩家HP恢复100点，获得稀有BOSS宝箱！", "Game Stats (HP / BOSS)"),
        ("蒙奇·D·路飞咧嘴大笑，振臂高呼：“我是要成为海贼王的男人！”", "Anime / Manga"),
        ("筑基初期与金丹大圆满，犹如云泥之别。", "Literary Cultivation Realms"),
    ]
    evaluate_sample_sentences(model, tokenizer, test_samples)

    print("\\n🎉 Training chunk finished successfully!")


if __name__ == "__main__":
    main()
'''

with open('/content/train_colab_zh_vi.py', 'w', encoding='utf-8') as f:
    f.write(code)

print('✅ Đã tạo /content/train_colab_zh_vi.py thành công với đầy đủ FFN fc1, fc2 & Beam Search!')

---
### 🟩 CELL 5: Sao Chép Nhanh Mô Hình Đã Merge Step 8000 Vào SSD Cục Bộ (Tăng tốc I/O)

In [ ]:
# CELL 5: Chuẩn bị Base Model Step 8000 từ Google Drive sang SSD cục bộ
import shutil

DRIVE_MERGED = '/content/drive/MyDrive/nllb_zh_vi_final_merged'
LOCAL_BASE = '/content/nllb_merged_base'

if not os.path.exists(LOCAL_BASE):
    if os.path.exists(DRIVE_MERGED):
        print(f'Đang copy mô hình đã merge từ {DRIVE_MERGED} sang {LOCAL_BASE}...')
        shutil.copytree(DRIVE_MERGED, LOCAL_BASE)
        print('✅ Đã nạp Base Model vào SSD cục bộ thành công!')
    else:
        print(f'⚠️ Không tìm thấy {DRIVE_MERGED} trên Google Drive! Hãy kiểm tra tên thư mục.')
else:
    print('✅ Base Model đã sẵn sàng trên SSD cục bộ!')

---
### 🟩 CELL 6: Huấn Luyện Chặng 1 (1.000 steps - LoRA v2 với FFN & Rank 32)

- Chạy 1.000 bước (32.000 câu).
- Mở khóa toàn diện: `q_proj, k_proj, v_proj, out_proj, fc1, fc2`.
- Tự động lưu checkpoint về Google Drive tại thư mục `/content/drive/MyDrive/nllb_zh_vi_checkpoints_v2`.

In [ ]:
# CELL 6: Chạy huấn luyện LoRA v2 Chặng 1
!python /content/train_colab_zh_vi.py \
    --model-path /content/nllb_merged_base \
    --train-file /content/dataset_full_zh_vi/train.jsonl \
    --eval-file /content/dataset_full_zh_vi/eval.jsonl \
    --output-dir /content/nllb_zh_vi_checkpoints_v2 \
    --drive-dir /content/drive/MyDrive/nllb_zh_vi_checkpoints_v2 \
    --max-steps 1000 \
    --batch-size 16 \
    --accum-steps 2 \
    --save-steps 250 \
    --lora-r 32 \
    --lora-alpha 64 \
    --target-modules q_proj k_proj v_proj out_proj fc1 fc2

---
### 🟩 CELL 7: Tiếp Tục Chạy Thêm Các Chặng Kế Tiếp (Cộng Dồn Tự Động)

- Khi muốn train tiếp, bạn chỉ cần chạy Cell này. Script sẽ tự động nhận diện checkpoint cũ trên Drive và train thêm +1.000 steps nữa (ví dụ: lên 2.000, 3.000, 4.000 steps...).
- Bạn có thể chạy cell này nhiều lần cho đến khi Loss hạ xuống khoảng **3.2 – 3.5**.

In [ ]:
# CELL 7: Chạy thêm chặng tiếp theo (+1.000 steps tự động cộng dồn)
!python /content/train_colab_zh_vi.py \
    --model-path /content/nllb_merged_base \
    --train-file /content/dataset_full_zh_vi/train.jsonl \
    --eval-file /content/dataset_full_zh_vi/eval.jsonl \
    --output-dir /content/nllb_zh_vi_checkpoints_v2 \
    --drive-dir /content/drive/MyDrive/nllb_zh_vi_checkpoints_v2 \
    --max-steps 1000 \
    --batch-size 16 \
    --accum-steps 2 \
    --save-steps 250 \
    --lora-r 32 \
    --lora-alpha 64 \
    --target-modules q_proj k_proj v_proj out_proj fc1 fc2

---
### 🟩 CELL 8: Đánh Giá Chất Lượng Trực Tiếp Bằng Beam Search (`num_beams=4`)

In [ ]:
# CELL 8: Đánh giá chất lượng dịch thuật trực tiếp trong Colab
import torch
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BASE_PATH = '/content/nllb_merged_base'
LORA_PATH = '/content/nllb_zh_vi_checkpoints_v2'

# Tìm checkpoint mới nhất
ckpts = [d for d in os.listdir(LORA_PATH) if d.startswith('checkpoint-')]
ckpts.sort(key=lambda x: int(x.split('-')[-1]))
latest_ckpt = os.path.join(LORA_PATH, ckpts[-1]) if ckpts else LORA_PATH
print(f'Nạp adapter từ: {latest_ckpt}')

tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_PATH, torch_dtype=torch.float16, device_map='cuda')
eval_model = PeftModel.from_pretrained(base_model, latest_ckpt)
eval_model.eval()

tokenizer.src_lang = 'zho_Hans'
tgt_id = tokenizer.convert_tokens_to_ids('vie_Latn')

test_cases = [
    ('克莱恩·莫雷蒂站在贝克兰德潮湿阴冷的雨夜街道上，拉低了黑色礼帽的帽檐。', 'Tây phương (Klein Moretti / Backland)'),
    ('白小纯深吸一口气，开始运转长生功，体内灵气如江河奔涌。', 'Tiên hiệp tu chân'),
    ('【系统提示】：玩家HP恢复100点，获得稀有BOSS宝箱！', 'Chỉ số Game / Hệ thống'),
    ('蒙奇·D·路飞咧嘴大笑，振臂高呼：“我是要成为海贼王的男人！”', 'Anime / Manga'),
    ('成步堂龙一猛地拍向审判席桌面：“反对！这份解剖记录存在致命矛盾！”', 'Tên Nhật Bản (Phoenix Wright)'),
]

print('=' * 80)
print('KẾT QUẢ DỊCH VỚI BEAM SEARCH (num_beams=4, no_repeat_ngram=3)')
print('=' * 80)
for zh, desc in test_cases:
    inp = tokenizer(zh, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = eval_model.generate(
            **inp,
            forced_bos_token_id=tgt_id,
            max_new_tokens=128,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.15,
        )
    pred = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'[{desc}]')
    print(f'  ZH  : {zh}')
    print(f'  PRED: {pred}')
    print('-' * 80)

---
### 🟩 CELL 9: Merge LoRA v2 Vào Base Model & Xuất Bản Hoàn Chỉnh Sang Google Drive

- Khi Loss đã hạ xuống vùng 3.2 – 3.5 và kết quả dịch đạt yêu cầu, chạy cell này để hợp nhất LoRA v2 vĩnh viễn vào model.
- Mô hình cuối cùng sẽ lưu tại `/content/drive/MyDrive/nllb_zh_vi_final_v2_merged` sẵn sàng để xuất sang ONNX INT8.

In [ ]:
# CELL 9: Hợp nhất (Merge) LoRA v2 vĩnh viễn vào Base Model
import torch
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BASE_PATH = '/content/nllb_merged_base'
LORA_PATH = '/content/nllb_zh_vi_checkpoints_v2'
MERGED_OUT = '/content/drive/MyDrive/nllb_zh_vi_final_v2_merged'

# Tìm checkpoint mới nhất để merge
ckpts = [d for d in os.listdir(LORA_PATH) if d.startswith('checkpoint-')]
ckpts.sort(key=lambda x: int(x.split('-')[-1]))
latest_ckpt = os.path.join(LORA_PATH, ckpts[-1]) if ckpts else LORA_PATH
print(f'Nạp checkpoint mới nhất để merge: {latest_ckpt}')

print('1. Nạp Base Model Step 8000...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_PATH, torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(BASE_PATH)

print('2. Nạp và hợp nhất LoRA v2 weights...')
model = PeftModel.from_pretrained(base_model, latest_ckpt)
merged_model = model.merge_and_unload()

print(f'3. Đang lưu mô hình hoàn chỉnh vào {MERGED_OUT} trên Google Drive...')
merged_model.save_pretrained(MERGED_OUT)
tokenizer.save_pretrained(MERGED_OUT)

print('=' * 80)
print('🎉 CHÚC MỪNG! Mô hình LoRA v2 hoàn chỉnh đã được hợp nhất thành công!')
print(f'Đường dẫn lưu trên Google Drive: {MERGED_OUT}')
print('=' * 80)